# Quan les dades no venen netes

**Optativa d'Aprenentatge automàtic · DAM/DAW 2n**

Els cinc quaderns anteriors funcionen perquè Iris és un dataset de joguina:
sense valors que falten, tot numèric, tot net. `fit()` sempre funciona a la
primera. A la feina no tens tanta sort: les dades arriben amb valors que
falten, columnes de text, escales incoherents i files duplicades. I `fit()`
peta directament, o —pitjor— no peta i et dona un resultat que no vol dir
res.

Aquest quadern fa servir el dataset `load_wine` de scikit-learn (178 vins,
13 mesures químiques, 3 varietats), fet malbé **a posta i de manera
reproduïble**, i desat com un fitxer CSV a `dades/vins_bruts.csv`. Vosaltres
el carregareu amb `pd.read_csv`, exactament com carregaríeu un fitxer que us
ha passat un company de feina.

## Les eines que necessites

Aquest quadern demana aquestes eines. Aquí les tens llistades, no
encadenades: quina va abans de quina, i sobre quines columnes, és
exactament la feina dels exercicis.

- De pandas: `pd.read_csv`, `.isna()`, `.dtypes`, `.describe()`,
  `.duplicated()`, `.drop_duplicates()`, `.reset_index()`, `.dropna()` i
  `pd.get_dummies`.
- De `sklearn.impute`: `SimpleImputer`.
- De `sklearn.preprocessing`: `OneHotEncoder`.
- De `sklearn.compose`: `ColumnTransformer`.
- De `sklearn.pipeline`: `Pipeline`.
- Ja conegudes dels quaderns anteriors: `train_test_split`,
  `RandomForestClassifier`, `load_wine`.

**Quatre d'aquestes no s'han vist a classe**: `ColumnTransformer`,
`OneHotEncoder`, `SimpleImputer` i `drop_duplicates`. No les busquis als
quaderns anteriors, perquè no hi surten. Les tens explicades a
`EX_00_caixa_eines.ipynb`, cada una amb la sintaxi, què li has de donar, què
et torna i el parany que té.

## Els objectes que manegues, i què els pots demanar

Tot aquest quadern es fa remenant tres o quatre objectes. Aquí tens, de cada un, el
que faràs servir de veritat: els **atributs** (sense parèntesis) i els **mètodes**
(amb parèntesis).

**El `DataFrame` de pandas** — `vins`, `vins_sd`, `X`, `X_entrena`. És l'objecte
central del quadern, i el diagnòstic de l'exercici 1 és, literalment, preguntar-li
coses:
- `.shape` — la tupla (files, columnes). Atribut, sense parèntesis
- `.columns` — els noms de columna; `list(X.columns)` te'ls dona com a llista
- `.dtypes` — el tipus de **cada** columna. Una columna de text surt com a `object`,
  i és el que et diu quina columna farà petar el `.fit()`
- `.index` — les etiquetes de fila. Després d'un `.drop_duplicates()` queden amb
  forats: per això hi va el `.reset_index(drop=True)`
- `.isna()` — un `DataFrame` de la **mateixa forma**, tot booleans: `True` on hi ha
  un nul. Encadenat amb `.sum()` te'ls compta per columna, i amb `.sum().sum()` en
  total
- `.dropna()` — una còpia sense les files que tenen algun nul. Amb `subset=[...]`
  només mira aquelles columnes. **No modifica l'original**: si no en desem el
  resultat, no ha passat res
- `.fillna(valor)` — una còpia amb els nuls substituïts
- `.duplicated()` — una `Series` de booleans, una per fila: `True` si aquella fila ja
  havia sortit abans. `.sum()` te les compta
- `.drop_duplicates()` — una còpia sense les repetides
- `.drop(columns=["classe"])` — una còpia sense aquella columna. Sense
  `columns=`, `.drop()` pensa que li dones **files**
- `.describe()` — el resum estadístic de les columnes numèriques
- `.median()`, `.mean()`, `.head()`, `.reset_index(drop=True)`, `.copy()`
- `vins["origen"]` en treu una columna com a `Series`; `vins[["a", "b"]]` (dobles
  claudàtors) en treu un `DataFrame` amb dues columnes

**La `Series` de pandas** — una columna sola, `vins["origen"]`, `y`:
- `.value_counts()` — quants cops apareix cada valor, ordenats de més a menys
- `.unique()` — els valors diferents, sense comptar-los; `.nunique()` quants són
- `.isna().sum()`, `.dtype`, `.shape`, `.median()`, `.mean()`
- `.astype(...)` per canviar-ne el tipus

**El `SimpleImputer`** — no és un model, és un **transformador**:
- `.fit(X)`, `.transform(X)`, i els dos de cop amb `.fit_transform(X)`. No té
  `.predict()` ni `.score()`
- `.statistics_` — el valor (mitjana o mediana) que ha calculat **per a cada
  columna** i que fa servir per omplir. És exactament el que et demana la
  comprovació de fuita de l'exercici 6
- compte amb què et torna: un **`ndarray`** de NumPy, no un `DataFrame`. Els noms de
  columna es perden pel camí, i si els vols recuperar has de tornar a muntar el
  `DataFrame` tu

**L'`OneHotEncoder`** — el mateix patró de transformador:
- `.fit(X)`, `.transform(X)`, `.fit_transform(X)`
- `.categories_` — els valors diferents que ha trobat a cada columna que li has
  passat. És una llista d'arrays, un per columna
- `.get_feature_names_out()` — els noms de les columnes noves
  (`origen_Catalunya`...). Sense això no saps quina columna del resultat és quina
- `handle_unknown="ignore"` és el que evita que peti si a l'examen apareix un valor
  que no era a l'entrenament
- espera una entrada **de dues dimensions**: `vins_sd[["origen"]]` amb dobles
  claudàtors, no `vins_sd["origen"]`

**El `ColumnTransformer`** — el que reparteix cada grup de columnes al seu
transformador:
- `.fit(X, y)`, `.transform(X)`, `.fit_transform(X, y)`
- `.named_transformers_` — un diccionari amb cada branca pel nom que li has posat
  (`"num"`, `"cat"`). És com arribes a l'imputador de dins per demanar-li
  `.statistics_`
- `.transformers` és el que li has demanat tu; `.transformers_` (amb guió baix) és
  el que ha quedat després del `.fit()`
- `.get_feature_names_out()` — els noms de totes les columnes de sortida

**El `Pipeline`** — per fora es comporta com un model qualsevol, i per dins guarda
els passos:
- `.fit(X, y)`, `.predict(X)`, `.score(X, y)` — igual que qualsevol model
- `.named_steps` — un diccionari amb els passos pel nom que els has posat:
  `pipe.named_steps["preprocessat"]`, `pipe.named_steps["model"]`. Encadenat amb
  `.named_transformers_["num"]` arribes fins a l'imputador
- `.steps` — la llista de parelles (nom, objecte)
- quan crides `.fit()` del pipeline, cada pas s'ajusta amb la sortida de l'anterior
  i **només** amb les files que li has passat. Això és tot el valor del pipeline a
  l'exercici 6

**El `RandomForestClassifier`** — el model del quadern:
- `.fit(X, y)`, `.predict(X)`, `.predict_proba(X)`, `.score(X, y)`
- `.classes_`, `.n_features_in_`, `.feature_importances_` — els tres són atributs
  apresos, i no existeixen abans del `.fit()`

Dues regles que valen per a tot el quadern, perquè són les que fan petar el codi
sense que sembli que hi hagi res malament:

- **Un atribut va sense parèntesis i un mètode amb parèntesis.** `model.coef_` et
  dona els coeficients; `model.coef_()` dona `TypeError`, perquè no és una funció.
  I a l'inrevés: el `.describe` d'un DataFrame sense parèntesis no calcula res,
  t'ensenya el mètode.
- **El guió baix al final vol dir «això ho he après de les dades».** `.coef_`,
  `.feature_importances_` i `.classes_` **no existeixen abans de cridar `.fit()`**:
  si els demanes abans, salta un error. El mateix passa amb els atributs apresos
  de qualsevol transformador.

Per interrogar qualsevol d'aquests objectes pel teu compte —`type()`, `dir()`,
`objecte.metode?` i l'autocompletat amb TAB— tens
`FO_00_objectes_i_autocompletar.ipynb`.

In [ ]:
import os
import numpy as np
import pandas as pd

RUTA_CSV = "dades/vins_bruts.csv"

def fabrica_vins_bruts():
    """Recrea el CSV brut a partir del Wine net de scikit-learn.

    Es determinista (np.random.seed(42)): sempre fa malbe exactament les
    mateixes files i columnes.
    """
    from sklearn.datasets import load_wine

    np.random.seed(42)

    wine = load_wine(as_frame=True)
    df = wine.frame.copy().rename(columns={"target": "classe"})
    n = len(df)

    # 1) valors que falten, en tres columnes i amb percentatges diferents
    columnes_amb_nuls = {"alcohol": 0.10, "magnesium": 0.20, "proline": 0.15}
    for columna, percentatge in columnes_amb_nuls.items():
        n_nuls = int(round(n * percentatge))
        idx_nuls = np.random.choice(df.index, size=n_nuls, replace=False)
        df.loc[idx_nuls, columna] = np.nan

    # 2) una columna de text, assignada a l'atzar
    df["origen"] = np.random.choice(["Catalunya", "Rioja", "Douro"], size=n)

    # 3) una columna amb una escala absurda (com si l'haguessin mesurat en una altra unitat)
    df["nonflavanoid_phenols"] = df["nonflavanoid_phenols"] * 100000

    # 4) files duplicades
    idx_duplicar = np.random.choice(df.index, size=20, replace=False)
    df = pd.concat([df, df.loc[idx_duplicar]], ignore_index=True)

    return df


if os.path.exists(RUTA_CSV):
    vins = pd.read_csv(RUTA_CSV)
else:
    # a Colab (o si algu ha esborrat el fitxer) el fabriquem igual que a classe
    print("No trobo el CSV -- el fabrico com a classe (normal a Colab).")
    vins = fabrica_vins_bruts()

print(f"Files: {len(vins)}, columnes: {len(vins.columns)}")
vins.head()

## Exercici 1 — El diagnòstic

Abans de tocar res, fes l'inventari dels desperfectes. Cap dataset ve amb una
"llista de defectes" enganxada: l'has de trobar tu. Amb `vins`, respon:

1. Quantes files i columnes té.
2. Quants valors nuls hi ha per columna (`.isna().sum()`), i quines columnes
   en tenen.
3. Quines columnes **no** són numèriques (`.dtypes`).
4. Quantes files duplicades hi ha (`.duplicated().sum()`).

Extra: fes `.describe()` de les columnes numèriques. Et sembla estranya
l'escala (mínim/màxim) d'alguna columna?

**Com saps que ho has fet bé:** pots dir, sense tornar a mirar el codi, tres
desperfectes concrets d'aquest dataset, cadascun amb un número exacte (no
"alguns nuls", sinó "39 nuls a `magnesium`").

In [ ]:
# Exercici 1: diagnostic

# 1. quantes files i columnes te `vins`? (vins.shape)


# 2. quants nuls hi ha per columna? (vins.isna().sum())


# 3. quines columnes no son numeriques? (vins.dtypes)


# 4. quantes files duplicades hi ha? (vins.duplicated().sum())


# extra: vins.describe() de les columnes numeriques -- alguna escala et sembla estranya?

## Exercici 2 — Intenta-ho a la bruta

Prova d'entrenar directament amb `vins` tal com està: separa `X` (totes les
columnes excepte `classe`) i `y` (`classe`), parteix en entrenament/examen
amb `train_test_split(test_size=0.3, random_state=42, stratify=y)`, i entrena
un `RandomForestClassifier(random_state=42)`.

**Petarà, i això és a posta.** Embolcalla el `.fit()` amb `try/except`, i
al bloc `except` imprimeix el tipus d'error i el seu missatge. Això forma
part de la lliçó: el missatge d'error diu exactament quin dels desperfectes
diagnosticats ha petat primer.

**Com saps que ho has fet bé:** t'ha sortit un missatge d'error imprès (no
una execució silenciosa), i el missatge assenyala una columna o un tipus de
dada concrets, no un error estrany sense relació.

In [ ]:
# Exercici 2: peta a proposit (forma part de l'exercici)
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

try:
    # separa X (vins.drop(columns=["classe"])) i y (vins["classe"])

    # parteix amb train_test_split(X, y, test_size=0.3, random_state=42,
    # stratify=y)

    # entrena un RandomForestClassifier(random_state=42) amb
    # .fit(X_entrena, y_entrena) -- es aqui on peta

    pass
except Exception as e:
    # imprimeix el tipus d'error (type(e).__name__) i el missatge (e)
    pass

## Exercici 3 — Els duplicats

Elimina les files duplicades (`.drop_duplicates()`), reinicia l'índex
(`reset_index(drop=True)`), i desa el resultat a `vins_sd` (*sense
duplicats*). Comprova quantes files hi havia abans i quantes en queden
després.

Raona, en un comentari o un `print`, **per què són perillosos** els
duplicats: si la mateixa fila acaba repartida entre entrenament i examen,
l'examen ja no examina res que el model no hagi vist.

**Com saps que ho has fet bé:** `vins_sd.duplicated().sum()` dona `0`, i pots
explicar en una frase per què un duplicat repartit entre entrenament i examen
invalida l'examen.

In [ ]:
# Exercici 3: duplicats

# elimina les files duplicades de `vins` amb .drop_duplicates() i desa el
# resultat a `vins_sd` (no oblidis reset_index(drop=True))

# quantes files hi havia abans i quantes despres de treure els duplicats?
# (len(vins) i len(vins_sd), o el seu .shape)

# comprova que vins_sd.duplicated().sum() es 0

## Exercici 4 — Els valors que falten

Treballa només amb les columnes numèriques (deixa `origen` per a
l'exercici següent); l'etiqueta és `classe`. Compara tres estratègies:

- **a)** Elimina les files amb algun nul (`.dropna()`). Quantes files et
  queden?
- **b)** Omple els nuls amb la mitjana de cada columna
  (`SimpleImputer(strategy="mean")`).
- **c)** Omple els nuls amb la mediana (`SimpleImputer(strategy="median")`).

Per a cadascuna, entrena un `RandomForestClassifier(random_state=42)` (amb
`train_test_split(test_size=0.3, random_state=42, stratify=y)` sobre les
dades que resultin de l'estratègia) i anota files disponibles i precisió.

**Com saps que ho has fet bé:** la (b) i la (c) han d'acabar amb exactament
el mateix nombre de files que `vins_sd`; si et queden amb menys, l'imputador
no ha omplert tot el que havia d'omplir. La (a) n'ha de tenir menys, i per
tant examina un conjunt d'examen diferent del de les altres dues: quan
compares les tres precisions, digues sobre quantes files d'examen s'ha
mesurat cadascuna.

In [ ]:
# Exercici 4: valors que falten (tres estrategies)
from sklearn.impute import SimpleImputer

cols_numeriques = [c for c in vins_sd.columns if c not in ("classe", "origen")]

# cada estrategia es queda amb un joc de files diferent, aixi que cada una
# ha de treure la seva X (les cols_numeriques) i la seva y (la columna
# "classe") del seu propi DataFrame, no del de l'anterior

# a) dropna: vins_sd.dropna(subset=cols_numeriques), parteix amb
#    train_test_split(X, y, test_size=0.3, random_state=42, stratify=y),
#    entrena amb .fit() i mira la precisio amb .score().
#    quantes files es perden respecte a vins_sd?


# b) SimpleImputer(strategy="mean"): omple els nuls amb la mitjana
#    (.fit_transform sobre les columnes numeriques -- torna un ndarray, no un
#    DataFrame), entrena amb el mateix train_test_split(random_state=42) i
#    mira la precisio. Mira-li tambe .statistics_


# c) SimpleImputer(strategy="median"): el mateix pero amb la mediana


# compara: files disponibles i precisio de cadascuna de les tres estrategies

## Exercici 5 — La columna de text

Els models només entenen números; `origen` és text. Aplica `OneHotEncoder`
(o `pd.get_dummies`) sobre la columna `origen` de `vins_sd` i mira com una
sola columna es converteix en tres. Imprimeix els noms de les columnes noves
i la forma (`shape`) del resultat.

**Com saps que ho has fet bé:** `origen` es converteix en exactament 3
columnes (una per valor: "Catalunya", "Rioja", "Douro"), i cada fila té un 1
en una d'elles i 0 a la resta.

In [ ]:
# Exercici 5: la columna de text
from sklearn.preprocessing import OneHotEncoder

# aplica OneHotEncoder a la columna "origen" de vins_sd amb
# .fit_transform(vins_sd[["origen"]]) -- dobles claudators, vol dues
# dimensions (o fes-ho amb pd.get_dummies)

# imprimeix els noms de les columnes noves
# (codificador.get_feature_names_out(), i mira tambe .categories_)
# i la forma (.shape) del resultat

## Exercici 6 — El Pipeline

Refés tot el procés (imputar + codificar) amb `Pipeline` i
`ColumnTransformer`, en comptes de fer-ho a mà pas a pas. Per què importa?

**Si calcules la mitjana o la mediana per omplir els buits amb el conjunt
sencer abans de partir-lo en entrenament/examen, estàs filtrant informació
de l'examen a l'entrenament**: la mitjana ja "coneix" valors que a l'examen
haurien de ser secrets. Un `Pipeline` ho fa en l'ordre correcte: s'ajusta
(`fit`) **només** amb l'entrenament, i després s'aplica (`transform`) a
l'examen sense tornar-se a ajustar.

Construeix:

1. Un `ColumnTransformer` amb dues branques: `SimpleImputer(strategy="median")`
   per a les columnes numèriques, i `OneHotEncoder(handle_unknown="ignore")`
   per a `origen`.
2. Un `Pipeline`: `[("preprocessat", el_column_transformer), ("model",
   RandomForestClassifier(random_state=42))]`.
3. Parteix `vins_sd` en entrenament/examen **abans** de fer res més, entrena
   el pipeline sencer amb `pipe.fit(X_entrena, y_entrena)`, i avalua'l amb
   `.score()`.
4. Comprova al teu propi codi que no hi ha fuita. Agafa l'imputador ja
   ajustat (`pipe.named_steps["preprocessat"].named_transformers_["num"]`) i
   compara els seus `.statistics_` amb la mediana de
   `X_entrena[cols_numeriques]` i amb la mediana de `X[cols_numeriques]`
   sencer. Són dues medianes diferents, i quina de les dues t'ha quedat
   desada a dins del pipeline diu amb quines files s'ha ajustat de veritat.

**Com saps que ho has fet bé:** en tot l'exercici no crides `.fit()` ni
`.fit_transform()` fora del pipeline abans de partir les dades; l'únic ajust
és `pipe.fit(X_entrena, y_entrena)`. I la comprovació del punt 4 ha de
coincidir exactament amb la mediana de `X_entrena[cols_numeriques]`, i **no**
amb la del conjunt sencer. Si et coincideix amb la del conjunt sencer, has
ajustat alguna cosa abans de partir: el `.score()` que et surti és més bo
del que et toca, i no perquè el model sigui millor.

In [ ]:
# Exercici 6: Pipeline + ColumnTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

X = vins_sd.drop(columns=["classe"])
y = vins_sd["classe"]

# parteix ABANS de fer res mes: train_test_split(X, y, test_size=0.3,
# random_state=42, stratify=y)

cols_numeriques = [c for c in X.columns if c != "origen"]

# construeix el ColumnTransformer:
#   ("num", SimpleImputer(strategy="median"), cols_numeriques)
#   ("cat", OneHotEncoder(handle_unknown="ignore"), ["origen"])

# construeix el Pipeline: [("preprocessat", ...), ("model", RandomForestClassifier(random_state=42))]

# entrena el pipeline NOMES amb dades d'entrenament (pipe.fit(X_entrena, y_entrena))
# i avalua amb pipe.score(X_examen, y_examen)


# comprovacio de fuita: agafa l'imputador ja ajustat
#   pipe.named_steps["preprocessat"].named_transformers_["num"]
# i compara els seus .statistics_ amb X_entrena[cols_numeriques].median()
# i amb X[cols_numeriques].median() sencer (np.allclose(...) et diu amb
# quina de les dues coincideix)

## Exercici 7 — El resultat final

Compara la precisió del teu pipeline (exercici 6) amb la d'un
`RandomForestClassifier(random_state=42)` entrenat sobre el `load_wine`
**net i original** de scikit-learn (sense cap desperfecte), amb el mateix
`train_test_split(test_size=0.3, random_state=42, stratify=y)`.

Les dues precisions no es mesuren sobre les mateixes files: `vins_sd` té els
duplicats fora i el `load_wine` original no, així que abans de comparar-les
apunta quantes files d'examen té cadascuna. Si et surt una diferència gran,
el sospitós no és el bosc: és alguna cosa que has fet pel camí (una
estratègia mal triada, una fuita d'informació, un error).

**Com saps que ho has fet bé:** les dues avaluacions han de compartir
`test_size`, `random_state` i el mateix model amb els mateixos
hiperparàmetres; l'única diferència entre totes dues han de ser les dades.
Escriu la diferència de precisió que t'ha sortit i sobre quantes files
d'examen s'ha mesurat cada una: un punt de diferència sobre cinquanta files
d'examen és una sola fila mal classificada, i d'una fila no se'n treu cap
conclusió.

In [ ]:
# Exercici 7: el resultat final
from sklearn.datasets import load_wine

# carrega load_wine(as_frame=True) i separa X i y: pots fer-ho amb .data i
# .target del Bunch, o amb .frame.drop(columns=["target"]) i .frame["target"]

# parteix amb el mateix train_test_split(test_size=0.3, random_state=42, stratify=y)

# entrena un RandomForestClassifier(random_state=42) amb .fit() i mira la
# seva precisio amb .score(). Apunta len(y_examen) de cada avaluacio

# compara-la amb la precisio del pipeline de l'exercici 6

## Resum

- Les dades brutes trenquen `fit()` de dues maneres: **petant** (columnes de
  text, valors `NaN`) o, més perillós, **no petant** i donant un resultat que
  no vol dir res.
- El primer pas sempre és **diagnosticar**: files, columnes, nuls per
  columna, tipus de dada, duplicats. No es pot arreglar el que no s'ha
  mirat.
- Els **duplicats** s'han d'eliminar abans de partir en entrenament/examen:
  si no, la mateixa fila pot acabar als dos costats i l'examen deixa de
  ser-ho.
- No hi ha una única manera correcta d'omplir **valors que falten**:
  eliminar files hi perd informació, la mitjana i la mediana en conserven
  més però n'inventen una part. Cal comparar-les.
- Els models només entenen números: una columna de **text** s'ha de
  codificar (`OneHotEncoder` o `pd.get_dummies`) abans d'entrenar.
- **`Pipeline` + `ColumnTransformer`** no és només comoditat: és l'única
  manera de garantir que la imputació i la codificació s'ajusten amb
  l'entrenament i s'apliquen (sense reajustar-se) a l'examen. Fer-ho a mà en
  l'ordre equivocat filtra informació de l'examen cap a l'entrenament, i el
  resultat sembla millor del que realment és.